# Imports

In [1]:
import sys

repo_path = '/Users/zohairshafi/Local Workspace/spectral_nature/'
sys.path.append(repo_path)
from src.utils.alpaca_utils import *
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_fscore_support
from sklearn.metrics import roc_auc_score, roc_curve, precision_recall_curve


from datetime import datetime
from multiprocessing import Pool, cpu_count
import multiprocessing as mp
import pickle as pkl
from functools import partial
import pandas as pd
import numpy as np
import os
import time

with open(os.path.join(cache_path, 'headers.pkl'), 'rb') as file: 
    headers, paper_header = pkl.load(file)



# Data Collection

In [ ]:
# Import the worker function from a top-level module so it's picklable by child processes
from src.utils.run_symbol_worker import run_symbol

symbol_list = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA', 'PINS', 'NVDA', 'QCOM']

# Progress logging setup
repo_path = '/Users/zohairshafi/Local Workspace/spectral_nature/'
progress_log_path = os.path.join(repo_path, 'cache', 'progress.log')
# Initialize/clear previous log
with open(progress_log_path, 'w') as _f:
    _f.write(f'Progress log started {datetime.utcnow().isoformat()}Z\n')

def log_progress(message: str):
    timestamp = datetime.utcnow().isoformat()
    with open(progress_log_path, 'a') as f:
        f.write(f'[{timestamp}] {message}\n')

# Ensure spawn start method (safer on macOS / notebooks)
try:
    mp.set_start_method('spawn', force=False)
except RuntimeError:
    # Already set in this interpreter
    pass

# Multiprocessing execution block
summaries = []

max_workers = min(8, cpu_count(), len(symbol_list))  # adjust as desired
log_progress(f'Launching process pool max_workers={max_workers} total_symbols={len(symbol_list)}')

with Pool(processes=max_workers) as pool:
    # Bind constant args for each worker using partial (picklable since function is module-level)
    worker_fn = partial(run_symbol, repo_path=repo_path, progress_log_path=progress_log_path)
    total = len(symbol_list)
    for completed, summary in enumerate(pool.imap_unordered(worker_fn, symbol_list), 1):
        summaries.append(summary)
        sym = summary.get('symbol', 'UNKNOWN')
        log_progress(f'COMPLETED symbol={sym} ({completed}/{total})')
        if completed % 2 == 0 or completed == total:
            pct = (completed/total)*100
            log_progress(f'OVERALL progress={completed}/{total} ({pct:0.1f}%)')

summary_df = pd.DataFrame(summaries)

display(summary_df)
print(f'Progress log written to: {progress_log_path}')

# Data Pre-Processing

In [ ]:
symbol_list = ['AAPL', 'MSFT', 'QCOM', 'GOOGL', 'AMZN', 'TSLA', 'PINS', 'NFLX', 'NVDA', 'META', 'SPY', 'QQQ']
data = []
for symbol in tqdm(symbol_list):
    with open(os.path.join(cache_path, 'closed_trades', f'{symbol}_closed_trades.pkl'), 'rb') as file: 
        closed_trades_df = pkl.load(file)
        closed_trades_df['Symbol'] = symbol
        data.append(closed_trades_df)
final_closed_trades_df = pd.concat(data).reset_index(drop=True)
final_closed_trades_df

100%|██████████| 11/11 [00:00<00:00, 60.58it/s]


,entry_date,entry_credit,entry_stock_price,strike,IV Short,IV Long,Delta_Short,Delta_Long,Gamma Short,Gamma Long,...,spread_distance,current_date,return_pct,short_exit_reason,short_exit_date,long_exit_reason,long_exit_date,exit_date,final_pnl,Symbol
0,2024-01-18 05:00:00+00:00,-267.0,188.70,190.0,0.279893,0.225208,0.472493,0.495347,0.038545,0.030355,...,21.0,2024-02-23 05:00:00+00:00,-1.460674,Short Expired,2024-02-02 00:00:00+00:00,Long Expired,2024-02-23 00:00:00+00:00,2024-02-23 05:00:00+00:00,-390.0,AAPL
1,2024-01-18 05:00:00+00:00,-189.0,188.70,195.0,0.267601,0.216161,0.284047,0.343305,0.034320,0.029147,...,21.0,2024-02-23 05:00:00+00:00,-1.190476,Short Expired,2024-02-02 00:00:00+00:00,Long Expired,2024-02-23 00:00:00+00:00,2024-02-23 05:00:00+00:00,-225.0,AAPL
2,2024-01-18 05:00:00+00:00,-162.0,188.70,180.0,0.293569,0.239996,-0.190367,-0.235592,0.025015,0.021924,...,21.0,2024-02-23 05:00:00+00:00,-1.697531,Short Expired,2024-02-02 00:00:00+00:00,Long Expired,2024-02-23 00:00:00+00:00,2024-02-23 05:00:00+00:00,-275.0,AAPL
3,2024-01-18 05:00:00+00:00,-117.0,188.70,175.0,0.308609,0.253223,-0.096354,-0.147773,0.014962,0.015590,...,21.0,2024-02-23 05:00:00+00:00,-1.410256,Short Expired,2024-02-02 00:00:00+00:00,Long Expired,2024-02-23 00:00:00+00:00,2024-02-23 05:00:00+00:00,-165.0,AAPL
4,2024-01-19 05:00:00+00:00,-225.0,191.55,195.0,0.269282,0.211442,0.383161,0.424646,0.039276,0.031737,...,21.0,2024-02-23 05:00:00+00:00,-1.160000,Short Expired,2024-02-02 00:00:00+00:00,Long Expired,2024-02-23 00:00:00+00:00,2024-02-23 05:00:00+00:00,-261.0,AAPL
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
882999,2025-03-07 05:00:00+00:00,-98.0,491.74,530.0,0.359225,0.198158,0.002587,0.076181,0.000613,0.005862,...,21.0,2025-03-31 04:00:00+00:00,-1.000000,Short Expired,2025-03-10 00:00:00+00:00,Long Expired,2025-03-31 00:00:00+00:00,2025-03-31 04:00:00+00:00,-98.0,QQQ
883000,2025-03-07 05:00:00+00:00,-154.0,491.74,525.0,0.317934,0.202269,0.002893,0.112302,0.000766,0.007651,...,21.0,2025-03-31 04:00:00+00:00,-1.000000,Short Expired,2025-03-10 00:00:00+00:00,Long Expired,2025-03-31 00:00:00+00:00,2025-03-31 04:00:00+00:00,-154.0,QQQ
883001,2025-03-07 05:00:00+00:00,-98.0,491.74,530.0,0.292925,0.198158,0.002590,0.076181,0.000614,0.005862,...,20.0,2025-03-31 04:00:00+00:00,-1.000000,Short Expired,2025-03-11 00:00:00+00:00,Long Expired,2025-03-31 00:00:00+00:00,2025-03-31 04:00:00+00:00,-98.0,QQQ
883002,2025-03-07 05:00:00+00:00,-240.0,491.74,520.0,0.296714,0.204873,0.005819,0.155739,0.001534,0.009462,...,21.0,2025-03-31 04:00:00+00:00,-1.004167,Short Expired,2025-03-10 00:00:00+00:00,Long Expired,2025-03-31 00:00:00+00:00,2025-03-31 04:00:00+00:00,-241.0,QQQ


In [ ]:
data = {
    'Delta Long' : final_closed_trades_df['Delta_Long'],
    'Delta Short' : final_closed_trades_df['Delta_Short'],
    'Delta Ratio' : final_closed_trades_df['Delta_Long'] / final_closed_trades_df['Delta_Short'],
    'Delta Difference' : final_closed_trades_df['Delta_Long'] - final_closed_trades_df['Delta_Short'],
    'Gamma Long' : final_closed_trades_df['Gamma Long'],
    'Gamma Short' : final_closed_trades_df['Gamma Short'],
    'Gamma Difference' : final_closed_trades_df['Gamma Long'] - final_closed_trades_df['Gamma Short'],
    'Gamma Ratio' : final_closed_trades_df['Gamma Long'] / final_closed_trades_df['Gamma Short'],
    'IV Long' : final_closed_trades_df['IV Long'],
    'IV Short' : final_closed_trades_df['IV Short'],
    'IV Difference' : final_closed_trades_df['IV Long'] - final_closed_trades_df['IV Short'],
    'IV Ratio' : final_closed_trades_df['IV Long'] / final_closed_trades_df['IV Short'],
    'Theta Long' : final_closed_trades_df['Theta Long'],
    'Theta Short' : final_closed_trades_df['Theta Short'],
    'Theta Difference' : final_closed_trades_df['Theta Long'] - final_closed_trades_df['Theta Short'],
    'Theta Ratio' : final_closed_trades_df['Theta Long'] / final_closed_trades_df['Theta Short'],
    'Vega Long' : final_closed_trades_df['Vega Long'],
    'Vega Short' : final_closed_trades_df['Vega Short'],
    'Vega Difference': final_closed_trades_df['Vega Long'] - final_closed_trades_df['Vega Short'],
    'Vega Ratio' : final_closed_trades_df['Vega Long'] / final_closed_trades_df['Vega Short'],
    'Forward Factor' : final_closed_trades_df['forward_factor'],
    'Spread' : final_closed_trades_df['spread_distance'],
    'Short DTE': (final_closed_trades_df['current_date'] - final_closed_trades_df['short_expiry']).dt.days,
    'Strike Deviation' : 100 * np.abs(final_closed_trades_df['entry_stock_price'] - final_closed_trades_df['strike']) / final_closed_trades_df['strike'],
    'PnL' : final_closed_trades_df['final_pnl'],
    'PnL Percent' : final_closed_trades_df['return_pct'],
    'Symbol' : final_closed_trades_df['Symbol'], 
    # 'Day of Year': final_closed_trades_df['entry_date'].dt.dayofyear,
}

data = pd.DataFrame(data)

In [ ]:
data

,Delta Long,Delta Short,Delta Ratio,Delta Difference,Gamma Long,Gamma Short,Gamma Difference,Gamma Ratio,IV Long,IV Short,...,Vega Short,Vega Difference,Vega Ratio,Forward Factor,Spread,Short DTE,Strike Deviation,PnL,PnL Percent,Symbol
0,0.495347,0.472493,1.048369,0.022854,0.030355,0.038545,-0.008190,0.787518,0.225208,0.279893,...,15.118549,8.453479,1.559146,55.725466,21.0,21,0.684211,-390.0,-1.460674,AAPL
1,0.343305,0.284047,1.208621,0.059258,0.029147,0.034320,-0.005173,0.849276,0.216161,0.267601,...,12.875957,8.855018,1.687717,54.150627,21.0,21,3.230769,-225.0,-1.190476,AAPL
2,-0.235592,-0.190367,1.237564,-0.045224,0.021924,0.025015,-0.003091,0.876444,0.239996,0.293569,...,10.320570,7.863194,1.761895,49.535743,21.0,21,4.833333,-275.0,-1.697531,AAPL
3,-0.147773,-0.096354,1.533642,-0.051419,0.015590,0.014962,0.000628,1.041955,0.253223,0.308609,...,6.487694,7.152692,2.102501,48.177034,21.0,21,7.828571,-165.0,-1.410256,AAPL
4,0.424646,0.383161,1.108272,0.041485,0.031737,0.039276,-0.007539,0.808052,0.211442,0.269282,...,14.212659,8.958227,1.630299,62.398371,21.0,21,1.769231,-261.0,-1.160000,AAPL
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
882999,0.076181,0.002587,29.453009,0.073595,0.005862,0.000613,0.005249,9.566936,0.198158,0.359225,...,0.344184,17.640312,52.252608,104.964739,21.0,21,7.218868,-98.0,-1.000000,QQQ
883000,0.112302,0.002893,38.818521,0.109409,0.007651,0.000766,0.006885,9.988094,0.202269,0.317934,...,0.380792,23.578996,62.920903,69.502587,21.0,21,6.335238,-154.0,-1.000000,QQQ
883001,0.076181,0.002590,29.417931,0.073591,0.005862,0.000614,0.005247,9.544218,0.198158,0.292925,...,0.401552,17.582944,44.787519,63.023174,20.0,20,7.218868,-98.0,-1.000000,QQQ
883002,0.155739,0.005819,26.761916,0.149919,0.009462,0.001534,0.007928,6.167144,0.204873,0.296714,...,0.711661,29.298336,42.168955,53.047531,21.0,21,5.434615,-241.0,-1.004167,QQQ


In [ ]:
# 1) Build features and target (binary: positive PnL)
feature_cols = [c for c in data.columns
                if c not in ['PnL', 'PnL Percent', 'Symbol']
                and pd.api.types.is_numeric_dtype(data[c])]

# Assemble modeling frame and clean
df_model = data[feature_cols + ['PnL']].copy()
df_model['label'] = (df_model['PnL'] > 0).astype(int)
df_model['regression_label'] = data['PnL Percent']
df_model = df_model.drop(columns=['PnL'])
df_model = df_model.replace([np.inf, -np.inf], np.nan).dropna()

X = df_model[feature_cols].to_numpy(dtype=np.float32)
y = df_model['label'].to_numpy(dtype=np.int64)
y_reg = df_model['regression_label'].to_numpy(dtype=np.float32)

test_size = 0.2
random_state = 42

X_train, X_test, y_train, y_test, y_reg_train, y_reg_test = train_test_split(X,
                                                    y,
                                                    y_reg,
                                                    test_size=test_size,
                                                    random_state=random_state,
                                                    stratify=y)


# 3) Standardize using train statistics (manual, no sklearn dependency)
mean_ = X_train.mean(axis = 0)
std_ = X_train.std(axis = 0) + 1e-8
X_train = (X_train - mean_) / std_
X_test = (X_test - mean_) / std_

# 4) Build PyTorch datasets and loaders
batch_size = 100000
train_ds = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train.astype(np.float32)), torch.from_numpy(y_reg_train.astype(np.float32)))
test_ds = TensorDataset(torch.from_numpy(X_test), torch.from_numpy(y_test.astype(np.float32)), torch.from_numpy(y_reg_test.astype(np.float32)))
train_loader = DataLoader(train_ds, batch_size = batch_size, shuffle = True)
test_loader = DataLoader(test_ds, batch_size = batch_size, shuffle = False)


torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
model = LogisticRegression(in_dim = X_train.shape[1]).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr = 1e-3)

# Handle class imbalance via pos_weight in BCEWithLogitsLoss
pos = float((y_train > 0.5).sum())
neg = float((y_train <= 0.5).sum())

pos_weight = torch.tensor([neg / pos], dtype=torch.float32, device=device)
criterion = nn.BCEWithLogitsLoss(pos_weight = pos_weight)

def mean_absolute_error(y_pred, y_true):
    return torch.mean(torch.abs(y_pred - y_true))

criterion_regression = mean_absolute_error


In [ ]:
epochs = 100
patience = 5  # epochs to wait for improvement
min_delta = 1e-4  # minimum change to qualify as improvement
best_val_loss = float('inf')
epochs_no_improve = 0
best_state = None

for epoch in range(1, epochs + 1):
    model.train()
    running_loss = 0.0
    for xb, yb, yb_reg in train_loader:
        
        xb = xb.to(device)
        yb = yb.to(device).flatten()
        yb_reg = yb_reg.to(device).flatten()

        optimizer.zero_grad()
        logits = model(xb)
        
        loss = criterion(logits[:, 0], yb)
        reg_loss = criterion_regression(logits[:, 1], yb_reg)

        total_loss = loss + reg_loss
        total_loss.backward()
        optimizer.step()

        running_loss += total_loss.item() * xb.size(0)

    epoch_loss = running_loss / len(train_loader.dataset)
    if epoch % 5 == 0 or epoch == 1:
        with torch.no_grad():
            logits = model(torch.from_numpy(X_train).to(device))[:, 0]
            preds = (torch.sigmoid(logits).cpu().numpy().ravel() >= 0.5).astype(int)
            acc = (preds == y_train).mean() if len(y_train) else float('nan')
        print(f'\nEpoch {epoch:03d} | Loss = {epoch_loss:.4f} | Train_acc = {acc:.3f}')

    # ----- Validation (for early stopping) -----
    model.eval()
    val_running_loss = 0.0
    acc = []
    with torch.no_grad():
        for xb, yb, yb_reg in test_loader:
            xb = xb.to(device)
            yb = yb.to(device).flatten()
            yb_reg = yb_reg.to(device).flatten()
            logits = model(xb)
            v_loss_cls = criterion(logits[:, 0], yb)
            v_loss_reg = criterion_regression(logits[:, 1], yb_reg)
            val_running_loss += (v_loss_cls + v_loss_reg).item() * xb.size(0)
            preds_test = (torch.sigmoid(logits[:, 0]).cpu().numpy().ravel() >= 0.5).astype(int)
            acc.append((preds_test == yb).mean() if len(yb) else float('nan'))
    val_loss = val_running_loss / len(test_loader.dataset)
    print (f'Epoch {epoch:03d} | Loss = {total_loss.item():.4f} | 
           Classification Loss: {loss.item():.4f} |
             Regression Loss: {reg_loss.item():.4f} | Test Accuracy: '{np.mean(np.array(acc))}, end='\r')


    # Check improvement
    if val_loss < (best_val_loss - min_delta):
        best_val_loss = val_loss
        best_state = copy.deepcopy(model.state_dict())
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f'Early stopping at epoch {epoch} (no improvement for {patience} epochs). Best val loss: {best_val_loss:.4f}')
            break

# Restore best model (if captured)
if best_state is not None:
    model.load_state_dict(best_state)

# 7) Evaluate on test set
model.eval()
with torch.no_grad():
    logits = model(torch.from_numpy(X_test).to(device))
    probs = torch.sigmoid(logits[:, 0]).cpu().numpy().ravel()
    y_pred = (probs >= 0.5).astype(int)
    y_reg_pred = logits[:, 1].cpu().numpy().ravel()
test_acc = (y_pred == y_test).mean() if len(y_test) else float('nan')
reg_test_mae = torch.mean(torch.abs(torch.tensor(y_reg_pred) - torch.tensor(y_reg_test))).item() if len(y_reg_test) else float('nan')
print(f'Test regression MAE: {reg_test_mae:.4f}')
print(f'Test accuracy: {test_acc:.3f}  (n_test={len(y_test)})')

# Precision, Recall, F1
prec, rec, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='binary', zero_division=0)
print(f'Precision: {prec:.3f}  Recall: {rec:.3f}  F1: {f1:.3f}')

# Show learned weights with feature names
coef = model.linear.weight[0].detach().cpu().numpy().ravel()
bias = float(model.linear.bias[0].detach().cpu().numpy().ravel())

coef_reg = model.linear.weight[1].detach().cpu().numpy().ravel()
bias_reg = float(model.linear.bias[1].detach().cpu().numpy().ravel())

coef_table = pd.DataFrame({'Features': feature_cols, 'Classification Weights': coef, 'Regression Weights': coef_reg}).sort_values('Classification Weights', ascending=False).reset_index(drop=True)
print('Bias:', bias)
print('Regression Bias:', bias_reg)


roc_auc = roc_auc_score(y_test, y_pred)
fpr, tpr, _ = roc_curve(y_test, y_pred)
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'ROC curve (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend()
plt.grid()
plt.show()

display(coef_table)

Epoch 001 | Loss = 2.6204 | Classification Loss: 1.0291 | Regression Loss: 1.5913
Epoch 001 | Loss = 2.6897 | Train_acc = 0.577
Epoch 001 | Loss = 2.6204 | Classification Loss: 1.0291 | Regression Loss: 1.5913
Epoch 001 | Loss = 2.6897 | Train_acc = 0.577
Validation loss: 2.6621
Validation loss: 2.6621
Validation loss: 2.620778 | Classification Loss: 1.0436 | Regression Loss: 1.6042
Validation loss: 2.6207
Validation loss: 2.583592 | Classification Loss: 1.0303 | Regression Loss: 1.5489
Validation loss: 2.5835
Validation loss: 2.550018 | Classification Loss: 1.0045 | Regression Loss: 1.5173
Validation loss: 2.5500
Epoch 005 | Loss = 2.5535 | Classification Loss: 1.0094 | Regression Loss: 1.5441
Epoch 005 | Loss = 2.5367 | Train_acc = 0.684
Epoch 005 | Loss = 2.5535 | Classification Loss: 1.0094 | Regression Loss: 1.5441
Epoch 005 | Loss = 2.5367 | Train_acc = 0.684
Validation loss: 2.5199
Validation loss: 2.5199
Validation loss: 2.492750 | Classification Loss: 0.9877 | Regression Loss:

Epoch 001 | Loss = 2.6204 | Classification Loss: 1.0291 | Regression Loss: 1.5913
Epoch 001 | Loss = 2.6897 | Train_acc = 0.577
Epoch 001 | Loss = 2.6204 | Classification Loss: 1.0291 | Regression Loss: 1.5913
Epoch 001 | Loss = 2.6897 | Train_acc = 0.577
Validation loss: 2.6621
Validation loss: 2.6621
Validation loss: 2.620778 | Classification Loss: 1.0436 | Regression Loss: 1.6042
Validation loss: 2.6207
Validation loss: 2.583592 | Classification Loss: 1.0303 | Regression Loss: 1.5489
Validation loss: 2.5835
Validation loss: 2.550018 | Classification Loss: 1.0045 | Regression Loss: 1.5173
Validation loss: 2.5500
Epoch 005 | Loss = 2.5535 | Classification Loss: 1.0094 | Regression Loss: 1.5441
Epoch 005 | Loss = 2.5367 | Train_acc = 0.684
Epoch 005 | Loss = 2.5535 | Classification Loss: 1.0094 | Regression Loss: 1.5441
Epoch 005 | Loss = 2.5367 | Train_acc = 0.684
Validation loss: 2.5199
Validation loss: 2.5199
Validation loss: 2.492750 | Classification Loss: 0.9877 | Regression Loss:

KeyboardInterrupt: 